# BetterTogether: Prompt + Weight Optimization (Chapter 6)

This notebook accompanies Chapter 6 §6.7 of *Context Engineering with DSPy*.

> **Requires a CUDA GPU for the automated path** (BetterTogether's `BootstrapFinetune` leg uses SGLang, which is CUDA-only). The notebook auto-detects SGLang support and falls back to a manual three-phase walkthrough (`p -> w -> p`) on non-CUDA machines so the prompt-optimization phases still run.


## Setup

In [ ]:
# Core dependencies
%pip install -r ../requirements.txt -q

# SGLang for full BetterTogether support (CUDA only)
# !pip install "sglang[all]>=0.4.4" -q


In [ ]:
import dspy
import torch
import random
import os
import time
import logging
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO)

print(f"DSPy version: {dspy.__version__}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
def get_device_info():
    """Detect available compute device."""
    if torch.cuda.is_available():
        device = "cuda"
        device_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ CUDA GPU: {device_name} ({vram_gb:.1f} GB VRAM)")
        sglang_supported = True
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = "mps"
        print("⚠️ Apple Silicon (MPS) — prompt optimization works, full BetterTogether requires CUDA")
        sglang_supported = False
    else:
        device = "cpu"
        print("⚠️ CPU only — prompt optimization works, full BetterTogether requires CUDA")
        sglang_supported = False
    return device, sglang_supported

DEVICE, SGLANG_SUPPORTED = get_device_info()

## Load Dataset

Same AI vs Human text detection task from the optimizer benchmark.

In [ ]:
import pandas as pd

csv_path = '../data/ai_vs_human200.csv'
df = pd.read_csv(csv_path)
examples = df.to_dict(orient='records')

dataset = [
    dspy.Example(**ex).with_inputs("text")
    for ex in examples
]

random.seed(42)
random.shuffle(dataset)

n = len(dataset)
train_end = int(n * 0.5)
val_end = int(n * 0.75)

trainset = dataset[:train_end]
valset = dataset[train_end:val_end]
testset = dataset[val_end:]

print(f"Training:   {len(trainset)} examples")
print(f"Validation: {len(valset)} examples")
print(f"Test:       {len(testset)} examples")

## Define Module and Metric

In [ ]:
class DetectAIText(dspy.Signature):
    """Analyze text and determine if it was written by an AI or a human."""
    text: str = dspy.InputField(description="The text to analyze")
    is_ai: bool = dspy.OutputField(description="True if AI-generated, False if human-written")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def exact_match(example, prediction, trace=None):
    return 1 if example.is_ai == prediction.is_ai else 0


print("Module: AIDetector (ChainOfThought)")
print("Metric: exact_match")

## How BetterTogether Works

BetterTogether orchestrates a multi-phase optimization strategy. The default strategy is `"p -> w -> p"`:

1. **Phase 1 (`p` — prompt optimization):** Optimize prompts using `BootstrapFewShotWithRandomSearch` on the base model. This finds the best instructions and few-shot examples.

2. **Phase 2 (`w` — weight optimization):** Use the prompt-optimized program to generate high-quality training traces, then fine-tune the student model on those traces using `BootstrapFinetune`.

3. **Phase 3 (`p` — re-optimize prompts):** Re-optimize prompts for the *fine-tuned* model. This is the key insight — the prompts that work best for a base model aren't the same as those that work best for a fine-tuned model. After fine-tuning, the model has internalized common patterns, so it typically needs fewer few-shot examples and different instructions.

The strategy string is configurable. You can run `"p"` (prompt only), `"w -> p"` (finetune then optimize), or any sequence of `p` and `w` phases.

## Option 1: Automated BetterTogether (CUDA Required)

On a machine with an NVIDIA GPU and SGLang installed, you can run BetterTogether as a single `compile()` call. This handles all three phases automatically.

In [ ]:
from dspy.clients.lm_local import LocalProvider

STUDENT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TEACHER_MODEL = "openai/gpt-4o-mini"

# Teacher LM (API-based)
teacher_lm = dspy.LM(
    TEACHER_MODEL,
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
    max_tokens=1000,
)

print(f"Teacher model: {TEACHER_MODEL}")
print(f"Student model: {STUDENT_MODEL}")
print(f"Device: {DEVICE}")

In [ ]:
if SGLANG_SUPPORTED:
    from dspy.teleprompt import BetterTogether, BootstrapFinetune, BootstrapFewShotWithRandomSearch

    # Must enable experimental mode
    dspy.settings.experimental = True

    # Set up student with local provider
    local_provider = LocalProvider()
    student_lm = dspy.LM(
        model=f"openai/local:{STUDENT_MODEL}",
        provider=local_provider,
        max_tokens=500,
    )

    # Create modules with assigned LMs
    student_module = AIDetector()
    student_module.set_lm(student_lm)

    # Configure sub-optimizers
    prompt_optimizer = BootstrapFewShotWithRandomSearch(
        metric=exact_match,
        max_bootstrapped_demos=8,
        num_candidate_programs=6,
        num_threads=4,
    )

    weight_optimizer = BootstrapFinetune(
        metric=exact_match,
        num_threads=4,
        train_kwargs={
            "num_train_epochs": 3,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 4,
            "learning_rate": 1e-5,
            "bf16": True,
            "use_peft": True,
        },
    )

    # Create BetterTogether optimizer
    optimizer = BetterTogether(
        metric=exact_match,
        p=prompt_optimizer,
        w=weight_optimizer,
    )

    print("Starting BetterTogether optimization...")
    print("Strategy: p -> w -> p (prompt, finetune, re-optimize)")

    start_time = time.time()
    optimized = optimizer.compile(
        student=student_module,
        trainset=trainset,
        strategy="p -> w -> p",
        valset_ratio=0.1,
    )
    elapsed = time.time() - start_time

    print(f"\nBetterTogether complete in {elapsed:.0f}s")

    # Evaluate
    evaluator = dspy.Evaluate(
        devset=testset,
        metric=exact_match,
        num_threads=4,
        display_progress=True,
        display_table=False,
    )
    score = evaluator(optimized)
    print(f"BetterTogether accuracy: {score:.1f}%")

else:
    print("SGLang not available — skipping automated BetterTogether.")
    print("See Option 2 below for a manual phase-by-phase approach,")
    print("or run this notebook on a CUDA-enabled machine.")

## Option 2: Manual Phase-by-Phase Approach

BetterTogether's `compile()` method orchestrates three phases automatically. But you can also run each phase manually, which gives you more control and works on any platform (you just need a fine-tuning provider for Phase 2).

This is actually how you'd use BetterTogether in production — running the phases separately lets you inspect intermediate results, use different optimizers per phase, and fine-tune through any provider (OpenAI API, local GPU, or a cloud service).

### Phase 1: Prompt Optimization on Base Model

First, optimize prompts against the base model. This finds the best instructions and few-shot examples for the *unmodified* model.

In [ ]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

# Configure the teacher/base model
dspy.configure(lm=teacher_lm)

# Baseline evaluation
baseline_evaluator = dspy.Evaluate(
    devset=testset,
    metric=exact_match,
    num_threads=4,
    display_progress=True,
    display_table=False,
)

print("Evaluating baseline (no optimization)...")
baseline_program = AIDetector()
baseline_score = baseline_evaluator(baseline_program)
print(f"Baseline accuracy: {baseline_score:.1f}%")

In [ ]:
# Phase 1: Optimize prompts with BootstrapFewShotWithRandomSearch
print("Phase 1: Optimizing prompts on base model...")

start_time = time.time()
prompt_optimizer = BootstrapFewShotWithRandomSearch(
    metric=exact_match,
    max_bootstrapped_demos=8,
    num_candidate_programs=6,
    num_threads=4,
)

phase1_program = prompt_optimizer.compile(
    AIDetector(),
    trainset=trainset,
    valset=valset,
)
phase1_time = time.time() - start_time

phase1_score = baseline_evaluator(phase1_program)
print(f"\nPhase 1 results:")
print(f"  Accuracy: {phase1_score:.1f}% (baseline: {baseline_score:.1f}%)")
print(f"  Uplift:   +{phase1_score - baseline_score:.1f}%")
print(f"  Time:     {phase1_time:.0f}s")

In [ ]:
# Inspect what Phase 1 found
for i, predictor in enumerate(phase1_program.predictors()):
    print(f"\nPredictor {i}: {predictor.__class__.__name__}")
    if hasattr(predictor, 'demos') and predictor.demos:
        print(f"  Few-shot demos: {len(predictor.demos)}")
    if hasattr(predictor, 'signature'):
        instructions = predictor.signature.instructions
        if instructions:
            print(f"  Instructions: {instructions[:200]}...")

### Phase 2: Fine-tune Student Model

Now use the prompt-optimized program to generate training traces, then fine-tune a smaller model on those traces. The optimized prompts from Phase 1 produce higher-quality traces than the baseline program would.

You have several options for fine-tuning:
- **Local GPU (CUDA):** Use DSPy's `BootstrapFinetune` directly
- **OpenAI API:** Export traces and fine-tune via `openai api fine_tunes.create`
- **Cloud providers:** Export traces for Anyscale, Together AI, etc.

Below we show both the DSPy automated path and a manual export approach.

In [ ]:
# Generate high-quality traces using the Phase 1 program
# These traces serve as training data for fine-tuning
print("Generating training traces from Phase 1 program...")

successful_traces = []
failed = 0

for i, example in enumerate(trainset):
    try:
        pred = phase1_program(text=example.text)
        score = exact_match(example, pred)
        if score:
            successful_traces.append({
                "text": example.text,
                "is_ai": example.is_ai,
                "reasoning": getattr(pred, 'reasoning', ''),
                "prediction": pred.is_ai,
            })
        else:
            failed += 1
    except Exception as e:
        failed += 1

    if (i + 1) % 25 == 0:
        print(f"  Processed {i + 1}/{len(trainset)} examples...")

print(f"\nTraining traces: {len(successful_traces)} successful, {failed} failed")
print(f"Success rate: {len(successful_traces) / len(trainset) * 100:.1f}%")

In [ ]:
if SGLANG_SUPPORTED:
    # Automated path: use BootstrapFinetune directly
    from dspy.teleprompt import BootstrapFinetune
    from dspy.clients.lm_local import LocalProvider

    local_provider = LocalProvider()
    student_lm = dspy.LM(
        model=f"openai/local:{STUDENT_MODEL}",
        provider=local_provider,
        max_tokens=500,
    )

    student_module = AIDetector()
    student_module.set_lm(student_lm)

    # Use the Phase 1 program as teacher
    teacher_module = phase1_program

    finetune_optimizer = BootstrapFinetune(
        metric=exact_match,
        num_threads=4,
        train_kwargs={
            "num_train_epochs": 3,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 4,
            "learning_rate": 1e-5,
            "bf16": True,
            "use_peft": True,
        },
    )

    print("Phase 2: Fine-tuning student model...")
    start_time = time.time()
    finetuned_module = finetune_optimizer.compile(
        student=student_module,
        teacher=teacher_module,
        trainset=trainset,
    )
    phase2_time = time.time() - start_time
    print(f"Phase 2 complete in {phase2_time:.0f}s")

else:
    print("BootstrapFinetune requires SGLang (CUDA).")
    print("\nTo fine-tune manually, export the traces above and use:")
    print("  - OpenAI fine-tuning API (for GPT models)")
    print("  - HuggingFace TRL/SFTTrainer (for open-source models)")
    print("  - Together AI or Anyscale (for cloud-hosted fine-tuning)")
    print(f"\nTraces available: {len(successful_traces)} examples")

In [ ]:
# Export traces for manual fine-tuning (works on any platform)
import json

output_path = Path("finetuned_models/bettertogether_traces.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    for trace in successful_traces:
        # Format as OpenAI fine-tuning JSONL
        entry = {
            "messages": [
                {
                    "role": "system",
                    "content": "Analyze text and determine if it was written by an AI or a human."
                },
                {
                    "role": "user",
                    "content": f"Text: {trace['text']}"
                },
                {
                    "role": "assistant",
                    "content": f"Reasoning: {trace['reasoning']}\nIs AI: {trace['prediction']}"
                }
            ]
        }
        f.write(json.dumps(entry) + "\n")

print(f"Exported {len(successful_traces)} traces to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print("\nUse this file with:")
print('  openai api fine_tuning.jobs.create -t bettertogether_traces.jsonl -m gpt-4o-mini-2024-07-18')

### Phase 3: Re-optimize Prompts for Fine-tuned Model

This is the step that makes BetterTogether more than just "prompt optimization + fine-tuning." After fine-tuning, the model has internalized patterns from the training data. The prompts that worked for the base model are now suboptimal — the fine-tuned model typically needs fewer few-shot examples (the patterns are already in its weights) and benefits from different instructions.

Re-optimizing finds this new optimum.

In [ ]:
if SGLANG_SUPPORTED:
    # Re-optimize prompts for the fine-tuned model
    # Use fewer demos — the model already knows common patterns
    print("Phase 3: Re-optimizing prompts for fine-tuned model...")

    phase3_optimizer = BootstrapFewShotWithRandomSearch(
        metric=exact_match,
        max_bootstrapped_demos=4,  # Fewer demos needed post-finetuning
        num_candidate_programs=6,
        num_threads=4,
    )

    start_time = time.time()
    final_program = phase3_optimizer.compile(
        finetuned_module.reset_copy(),
        trainset=trainset,
        valset=valset,
    )
    phase3_time = time.time() - start_time

    # Evaluate final result
    final_score = baseline_evaluator(final_program)
    print(f"\nPhase 3 results:")
    print(f"  Accuracy:  {final_score:.1f}%")
    print(f"  Time:      {phase3_time:.0f}s")
    print(f"\n--- BetterTogether Summary ---")
    print(f"  Baseline:   {baseline_score:.1f}%")
    print(f"  Phase 1 (p): {phase1_score:.1f}%")
    print(f"  Phase 3 (p->w->p): {final_score:.1f}%")
    print(f"  Total uplift: +{final_score - baseline_score:.1f}%")

else:
    print("Phase 3 requires the fine-tuned model from Phase 2.")
    print("\nOnce you have a fine-tuned model (from any provider), configure it:")
    print()
    print('  # For OpenAI fine-tuned model:')
    print('  finetuned_lm = dspy.LM("ft:gpt-4o-mini:my-org:my-model:abc123")')
    print('  dspy.configure(lm=finetuned_lm)')
    print()
    print('  # Re-optimize with fewer demos')
    print('  phase3_optimizer = BootstrapFewShotWithRandomSearch(')
    print('      metric=exact_match,')
    print('      max_bootstrapped_demos=4,  # Fewer demos needed post-finetuning')
    print('  )')
    print('  final_program = phase3_optimizer.compile(AIDetector(), trainset=trainset, valset=valset)')

## Results Summary

Here are the results from Phase 1 (prompt optimization on the base model). Phases 2-3 require CUDA infrastructure — see the chapter text for expected results from the full BetterTogether pipeline.

In [ ]:
print("=" * 55)
print("BetterTogether — Phase 1 Results (Prompt Optimization)")
print("=" * 55)
print(f"  Baseline accuracy:     {baseline_score:.1f}%")
print(f"  Phase 1 accuracy:      {phase1_score:.1f}%")
print(f"  Uplift:                +{phase1_score - baseline_score:.1f}%")
print(f"  Time:                  {phase1_time:.0f}s")
print()

if SGLANG_SUPPORTED:
    print("Full BetterTogether Results (p -> w -> p)")
    print("=" * 55)
    print(f"  Phase 1 (p):       {phase1_score:.1f}%")
    print(f"  Phase 3 (p->w->p): {final_score:.1f}%")
    print(f"  Total uplift:      +{final_score - baseline_score:.1f}%")
else:
    print("Phases 2-3 require CUDA + SGLang.")
    print("Expected improvement from full BetterTogether: +5-10% over Phase 1 alone.")

## Summary

**BetterTogether orchestrates prompt optimization and fine-tuning in a synergistic workflow:**

1. Optimize prompts for the base model
2. Generate training data from the optimized program
3. Fine-tune a student model on those traces
4. Re-optimize prompts for the fine-tuned model

**The key insight:** prompts that work for a base model aren't optimal for a fine-tuned model. Re-optimizing after fine-tuning finds a new, better equilibrium.

**When to use BetterTogether:**
- Mission-critical deployments where every accuracy point matters
- High volume (100k+ requests/month) to amortize fine-tuning cost
- You've plateaued with prompt optimization alone

**When to skip it:**
- Prototyping (use BootstrapFewShot)
- No fine-tuning infrastructure or budget
- Prompt optimization alone meets your accuracy target